# TokenOS serving benchmark (Part B)
Replays real TokenOS retry traffic, full history vs adaptive, against a self-hosted vLLM server on a T4, one optimization at a time. See `README.md`. Results go to `results/serving.json`.

In [ ]:
# Runtime > Change runtime type > T4 GPU first.
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone --depth 1 -b bigcodebench-evidence-and-preflight https://github.com/default2543/tokenOS.git
%cd tokenOS/colab

vLLM 0.6.6 (pinned: its V0 engine supports Turing) needs Python 3.12 or older, and Colab now runs 3.13. This cell installs it into a separate Python 3.12 environment with `uv`, leaving Colab's own Python alone, so **no restart is needed**. Every later cell runs with that environment's Python. Takes about 3-5 minutes.

In [ ]:
!pip install -q uv
!uv venv -q --python 3.12 /content/vllm-env
# transformers is pinned: newer releases need PyTorch >= 2.6 and disable model loading on vLLM 0.6.6's 2.5.1.
!uv pip install -q --python /content/vllm-env/bin/python vllm==0.6.6.post1 "transformers==4.47.1" numpy pytest
# Should print vLLM 0.6.6.post1 and the GPU name.
!/content/vllm-env/bin/python -c "import vllm, torch, triton; print(vllm.__version__, torch.__version__, triton.__version__, torch.cuda.get_device_name())"

## Kernels vs vLLM's PagedAttention (about 5 minutes)
Times the Triton decode kernels against the decode kernel vLLM itself uses on this GPU, and confirms the tuned split-K defaults. You can stop after this cell if you only want the kernel comparison.

In [ ]:
%cd /content/tokenOS/colab/kernels
!/content/vllm-env/bin/python bench.py --vllm --out ../results/kernels-vllm.json
from google.colab import files
files.download('../results/kernels-vllm.json')

In [ ]:
%cd /content/tokenOS/colab
# Smoke test: one task, baseline only. Fix any error here before the long runs.
!/content/vllm-env/bin/python serving_bench.py --workload workloads/bcb-hard-8039bc.jsonl.gz --configs baseline --tasks 1 --out results/smoke.json

## B2.1–B2.3: baseline, + prefix caching, + n-gram speculative decoding (Qwen2.5-Coder-7B AWQ)
About 40 minutes.

In [ ]:
!/content/vllm-env/bin/python serving_bench.py --workload workloads/bcb-hard-8039bc.jsonl.gz --configs baseline,prefix-cache,prefix-cache+ngram --out results/serving.json

## B2.4: quantization, 1.5B fp16 vs AWQ
If the AWQ checkpoint name fails to download, check the exact name on Hugging Face and edit `SMALL_AWQ` in `serving_bench.py`.

In [ ]:
!/content/vllm-env/bin/python serving_bench.py --workload workloads/bcb-hard-8039bc.jsonl.gz --configs small-fp16,small-awq --out results/serving-quant.json

## B2.5: concurrency sweep on the best 7B configuration

In [ ]:
!/content/vllm-env/bin/python serving_bench.py --workload workloads/bcb-hard-8039bc.jsonl.gz --configs prefix-cache --concurrency-sweep 1,4,16 --out results/serving-concurrency.json

In [ ]:
!cd results && zip -q -r serving-results.zip . && ls -la
from google.colab import files
files.download('results/serving-results.zip')